# 코퍼스 — raw/를 훑어 text/와 장부를 세운다


In [ ]:
import sys
from pathlib import Path

_starts = [Path.cwd().resolve()]
_nb = globals().get("__vsc_ipynb_file__")
if isinstance(_nb, str):
    _starts.append(Path(_nb).resolve().parent)
_root = None
for _start in _starts:
    for _p in [_start, *_start.parents]:
        if (_p / "paths.py").is_file() and (_p / "day01").is_dir():
            _root = _p
            break
    if _root is not None:
        break
if _root is None:
    _root = Path.home() / "p3-llm" / "c4-data"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from paths import CORPUS, RAW, TEXT, PDF, MANIFEST, load_api_env

load_api_env()
root = CORPUS
TEXT.mkdir(parents=True, exist_ok=True)
print("CORPUS", CORPUS)
print("PDF.exists", PDF.exists())

import json
import shutil
from datetime import date

import pandas as pd
import pymupdf
import tiktoken
from bs4 import BeautifulSoup
from markdownify import markdownify as to_md

encoder = tiktoken.get_encoding("o200k_base")


## `build()` — raw/를 훑어 text/와 장부를 세운다

원본은 지우지 않는다. 출처·라이선스·추출기·쪽별 문자수·수집일을 `corpus/manifest.json`에 사람이 읽을 수 있게 적는다. `source`를 `?`로 두면 오늘 과제를 안 한 것이다.


In [ ]:
import json
import shutil
from datetime import date

META = {
    "paper-wheat-dss.pdf": {
        "source": "한국농공학회논문집 66(4) 2024, 김솔희 외. 수업 내 인용.",
        "license": "수업 내 인용",
    },
    "wiki-llm.html": {
        "source": "https://ko.wikipedia.org/wiki/대형_언어_모델",
        "license": "CC BY-SA 4.0",
    },
    "schedule.csv": {
        "source": "생성형 AI 과정 시간표 배포판",
        "license": "과정 내부 배포",
    },
    "guide-day-prev.md": {
        "source": "전일 학습자 가이드 (2026-08-18 구조화 출력과 도구 호출)",
        "license": "수업 자료",
    },
    "pipa.md": {
        "source": "개인정보 보호법 전문 (법제처 / 위키문헌)",
        "license": "저작권법 제7조 자유이용",
    },
}


def extract_pdf(path: Path):
    d = pymupdf.open(path)
    page_chars = [len(p.get_text() or "") for p in d]
    body = "\n\n".join(p.get_text() or "" for p in d)
    return body, page_chars, "pymupdf"


def extract_html(path: Path):
    soup = BeautifulSoup(path.read_text(encoding="utf-8", errors="ignore"), "lxml")
    body = soup.select_one("#mw-content-text") or soup.body or soup
    text = to_md(str(body), heading_style="ATX", strip=["a", "img", "sup"])
    return text, [len(text)], "beautifulsoup4+lxml+markdownify"


def extract_csv(path: Path):
    frame = pd.read_csv(path, encoding="cp949")
    lines = []
    for _, row in frame.iterrows():
        parts = [f"{col}은 {row[col]}" for col in frame.columns if pd.notna(row[col])]
        lines.append("。 ".join(parts) + ".")
    text = "\n".join(lines)
    return text, [len(text)], "pandas/cp949"


def extract_md(path: Path):
    text = path.read_text(encoding="utf-8")
    return text, [len(text)], "plain-utf8"


def extract_one(path: Path):
    suf = path.suffix.lower()
    if suf == ".pdf":
        return extract_pdf(path)
    if suf in {".html", ".htm"}:
        return extract_html(path)
    if suf == ".csv":
        return extract_csv(path)
    if suf in {".md", ".txt"}:
        return extract_md(path)
    raise ValueError(f"모름: {path.name}")


def build():
    TEXT.mkdir(parents=True, exist_ok=True)
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for path in sorted(RAW.iterdir()):
        if not path.is_file() or path.name.startswith("."):
            continue
        body, page_chars, extractor = extract_one(path)
        out = TEXT / (path.stem + ".txt")
        out.write_text(body, encoding="UTF-8")
        info = META.get(path.name, {"source": path.name, "license": "?"})
        rec = {
            "file": path.name,
            "text": out.name,
            "source": info["source"],
            "license": info["license"],
            "extractor": extractor,
            "chars": len(body),
            "page_chars": page_chars,
            "collected": date.today().isoformat(),
        }
        rec["tokens"] = len(encoder.encode(body))
        rows.append(rec)
    MANIFEST.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows


MANIFEST = (root / "corpus" / "manifest.json") if (root / "corpus").is_dir() else (root / "manifest.json")
print("raw", RAW.resolve())
print("text", TEXT.resolve())
print("manifest", MANIFEST.resolve())


In [ ]:
ledger = build()
for rec in ledger:
    print(f"{rec['file']:22} {rec['chars']:7d}자  {rec['tokens']:6d}토큰  {rec['source'][:40]}")
print("합계 글자", sum(r["chars"] for r in ledger), "토큰", sum(r["tokens"] for r in ledger))


## 재현성 — text/를 지워도 장부가 다시 선다

손으로 고친 파일이 `text/`에만 있으면 지우는 순간 사라진다. 원본은 `raw/`에 두고 `build()`만 다시 돌린다.


In [ ]:
before = {p.name: p.read_bytes() for p in TEXT.glob("*.txt")}
shutil.rmtree(TEXT)
ledger2 = build()
after = {p.name: p.read_bytes() for p in TEXT.glob("*.txt")}
print("파일 수", len(before), "→", len(after))
print("바이트 일치", before == after)
print("장부 줄", len(ledger2))


## 코퍼스 토큰 · 임베딩 비용 어림

단가는 기억하지 말고 공식 문서를 본다. `text-embedding-3-small` 입력은 2026-08 기준 **$0.020 / 1M 토큰** (https://platform.openai.com/docs/pricing).


In [ ]:
total_tokens = sum(r["tokens"] for r in ledger2)
usd_per_m = 0.020
cost = total_tokens / 1_000_000 * usd_per_m
print(f"토큰 {total_tokens:,}")
print(f"임베딩 1회 어림 ${cost:.6f}  (단가 ${usd_per_m}/1M, 공식 가격표)")


## 메이킹 — 내 문서 한 줄

`raw/`에 파일을 하나 더 넣고 `META`에 출처·라이선스를 적은 뒤 `build()`를 다시 돌린다. `source`를 `?`로 남기지 않는다. 스캔 PDF면 `chars: 0`이 장부가 알려 주는 것이다.
